# RL Teaching — Day 1
**Bandit · FrozenLake Q-Learning · Q-Table Visualization**

> 📌 **Before you start:** Go to **File → Save a copy in Drive** to make your own editable copy.

You don't need to understand every line of code.
Focus on **what changes when you adjust the 🔧 parameters** and re-run the cell.

In [ ]:
# ── Setup (run this first) ──────────────────────────────────
!pip install gymnasium matplotlib seaborn numpy --quiet
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print('✅ Setup complete!')

---
## Part 1 · Multi-Armed Bandit

Imagine a row of slot machines. Each machine has a *hidden* probability of giving a reward.
The agent uses **ε-greedy strategy**:
- With probability **ε** → try a random machine *(explore)*
- With probability **1 - ε** → pick the best-known machine *(exploit)*

### 🔧 Your task
Run the cell with **ε = 0.1**, then change it to **ε = 0.9** and run again.
What is different about the two reward curves?

In [ ]:
# ════════════════════════════════════════════
epsilon = 0.1   # 🔧 Try: 0.1 / 0.5 / 0.9
# ════════════════════════════════════════════

np.random.seed(42)
n_bandits, n_rounds = 5, 500
true_probs = [0.2, 0.5, 0.8, 0.4, 0.6]   # hidden from the agent

Q = np.zeros(n_bandits)
N = np.zeros(n_bandits)
rewards = []

for t in range(n_rounds):
    action = np.random.randint(n_bandits) if np.random.random() < epsilon else np.argmax(Q)
    reward = 1 if np.random.random() < true_probs[action] else 0
    N[action] += 1
    Q[action] += (reward - Q[action]) / N[action]
    rewards.append(reward)

smoothed = np.convolve(rewards, np.ones(30)/30, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(smoothed, linewidth=2, label=f'epsilon = {epsilon}')
plt.axhline(max(true_probs), color='gray', linestyle='--', label='Best possible reward')
plt.xlabel('Round'); plt.ylabel('Avg Reward (smoothed)')
plt.title(f'epsilon-greedy Bandit  |  epsilon = {epsilon}')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"Agent best guess : Machine #{np.argmax(Q)} (Q = {max(Q):.2f})")
print(f"Actual best      : Machine #{np.argmax(true_probs)} (p = {max(true_probs):.2f})")

---
## Part 2 · FrozenLake — Q-Learning

FrozenLake is a 4x4 grid. The agent starts at **S** and must reach **G** without falling into holes **H**.
```
S  F  F  F
F  H  F  H
F  F  F  H
H  F  F  G
```
The agent learns using **Q-learning**: it builds a Q-table storing
how good each action is from each state.

### 🔧 Your task
Run with default settings first, then change **one parameter at a time** and observe the effect.

In [ ]:
# ════════════════════════════════════════════
alpha      = 0.8    # 🔧 learning rate    — try: 0.1 / 0.5 / 0.8
gamma      = 0.95   # 🔧 discount factor  — try: 0.5 / 0.95 / 0.99
epsilon    = 0.1    # 🔧 exploration rate — try: 0.05 / 0.1 / 0.5
n_episodes = 2000
# ════════════════════════════════════════════

env = gym.make('FrozenLake-v1', is_slippery=False)
Q   = np.zeros([env.observation_space.n, env.action_space.n])
episode_rewards, episode_lengths = [], []

for _ in range(n_episodes):
    state, _ = env.reset()
    total, steps, done = 0, 0, False
    while not done:
        action = env.action_space.sample() if np.random.random() < epsilon else np.argmax(Q[state])
        ns, r, ter, tru, _ = env.step(action)
        done = ter or tru
        Q[state, action] += alpha * (r + gamma * np.max(Q[ns]) - Q[state, action])
        state, total, steps = ns, total + r, steps + 1
    episode_rewards.append(total)
    episode_lengths.append(steps)

window = 100
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(np.convolve(episode_rewards, np.ones(window)/window, mode='valid'), color='steelblue', lw=2)
ax1.set(title=f'Reward | alpha={alpha} gamma={gamma} epsilon={epsilon}',
        xlabel='Episode', ylabel='Reward (1 = reached goal)')
ax1.grid(alpha=0.3)
ax2.plot(np.convolve(episode_lengths, np.ones(window)/window, mode='valid'), color='coral', lw=2)
ax2.set(title='Episode Length', xlabel='Episode', ylabel='Steps')
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Success rate (last 500): {np.mean(episode_rewards[-500:]):.1%}')

---
## Part 3 · Q-Table Visualization

The Q-table assigns a value to every (state, action) pair.
- **High value** → agent thinks this action is good from this state
- **Arrow** → preferred action in each cell

### 🔧 Your task
1. Can you trace a path from **S** to **G** following the arrows?
2. Change `state_to_inspect` to explore Q-values in different cells.

In [ ]:
# ════════════════════════════════════════════
state_to_inspect = 14   # 🔧 any cell 0-15 (0=top-left, 15=bottom-right)
# ════════════════════════════════════════════

symbols     = ['L', 'D', 'R', 'U']   # Left Down Right Up
best_arrows = np.vectorize(lambda x: symbols[x])(np.argmax(Q, axis=1).reshape(4,4))
max_Q_grid  = np.max(Q, axis=1).reshape(4,4)
special     = {0:'S', 5:'H', 7:'H', 11:'H', 12:'H', 15:'G'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(max_Q_grid, annot=best_arrows, fmt='', cmap='YlOrRd',
            linewidths=1, ax=ax1, mask=(max_Q_grid == 0),
            cbar_kws={'label': 'Max Q-value (confidence)'})
for pos, label in special.items():
    r, c = divmod(pos, 4)
    ax1.text(c+0.5, r+0.5, label, ha='center', va='center',
             fontsize=14, fontweight='bold',
             color='navy' if label in ('S','G') else 'red')
ax1.set(title='Best Action per State (L/D/R/U = direction, color = confidence)',
        xlabel='Column', ylabel='Row')

q_vals = Q[state_to_inspect]
bars = ax2.bar(symbols, q_vals, color=['#e74c3c','#2ecc71','#3498db','#f39c12'])
ax2.set(title=f'Q-values for State {state_to_inspect} (row {state_to_inspect//4}, col {state_to_inspect%4})',
        xlabel='Action', ylabel='Q-value')
ax2.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, q_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}',
             ha='center', va='bottom', fontsize=11)
plt.tight_layout(); plt.show()
print(f'State {state_to_inspect}: preferred action = {symbols[np.argmax(q_vals)]}')